In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Dataset
text = """
artificial intelligence systems learn patterns from data
sequence models process information step by step
recurrent neural networks are useful for sequence prediction
lstm networks handle long term dependencies
deep learning models improve sequence learning
generative models create new samples from learned patterns
language models predict the next word in a sentence
sequence generation is used in chatbots and assistants
machine learning helps computers learn automatically
training data improves model accuracy
neural networks simulate human brain structures
optimization algorithms improve learning efficiency
technology is transforming modern education
online learning platforms use artificial intelligence
students benefit from intelligent tutoring systems
automation improves productivity and decision making
"""

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

# Create input sequences
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i + 1]
        input_sequences.append(n_gram)

# Padding
max_seq_len = max(len(x) for x in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

# Split into X and y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# Convert output to categorical
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Build LSTM model
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=200, verbose=1)

# Text generation function
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text

# Generate output
print(generate_text("artificial intelligence", 5))

Epoch 1/200


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.0114 - loss: 4.3580  
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0341 - loss: 4.3468    
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0341 - loss: 4.3372    
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0568 - loss: 4.3247
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0455 - loss: 4.3060
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0455 - loss: 4.2794
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0455 - loss: 4.2340
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0455 - loss: 4.1634
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0455 - loss: 4.0970
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0455 - loss: 4.0824    
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0455 - loss: 4.0627    
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.068

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Dense, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model
from tensorflow.keras import Input
import numpy as np

# Dataset
sentences = [
    "artificial intelligence systems learn patterns from data",
    "sequence models process information step by step",
    "recurrent neural networks are useful for sequence prediction",
    "lstm networks handle long term dependencies",
    "deep learning models improve sequence learning",
    "generative models create new samples from learned patterns",
    "language models predict the next word in a sentence",
    "sequence generation is used in chatbots and assistants"
]

# Vectorization
vectorizer = TextVectorization(output_sequence_length=10)
vectorizer.adapt(sentences)

vocab_size = len(vectorizer.get_vocabulary())

# Convert text to sequences
sequences = vectorizer(sentences)

X = sequences[:, :-1]
y = sequences[:, 1:]

# Positional Encoding
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embeddings = Embedding(vocab_size, embed_dim)
        self.position_embeddings = Embedding(sequence_length, embed_dim)

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

# Transformer Block
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm = LayerNormalization()
        self.dense = Dense(embed_dim)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        x = self.norm(inputs + attn_output)
        return self.dense(x)

# Model
embed_dim = 64
inputs = Input(shape=(9,))
x = PositionalEmbedding(9, vocab_size, embed_dim)(inputs)
x = TransformerBlock(embed_dim, 2)(x)
outputs = Dense(vocab_size, activation='softmax')(x)

transformer_model = Model(inputs, outputs)

transformer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

transformer_model.fit(X, y, epochs=100)

print("Transformer model trained successfully!")

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 4.3424
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.0000e+00 - loss: 3.8900
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.0556 - loss: 3.4756
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 3.1089
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3194 - loss: 2.8005
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.3333 - loss: 2.5552
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.3611 - loss: 2.3638
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.3750 - loss: 2.2045
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3889 - loss: 2.0548
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.4167 - loss: 1.9040
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 1.7533
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.597